In [1]:
import numpy as np
import pandas as pd

from ecg_waveform.config import load_yaml
from ecg_waveform.core import Fiducials
from ecg_waveform.data import FiducialMapper, WFDBLoader
from ecg_waveform.detection import build_detector
from ecg_waveform.evaluation import compute_metrics
from ecg_waveform.signal.preprocessing import Preprocessor

preprocessor_config = load_yaml("preprocessing/preprocessing_pipeline.yaml")
preprocessor = Preprocessor.from_config(preprocessor_config)

mapper = FiducialMapper.from_yaml("mapper/annotation_labels.yaml")

loader = WFDBLoader("mitdb", annotation_extension="atr", symbol_mapper=mapper)

In [2]:
detector = build_detector("pan_tompkins_plus_plus")

results = []
for record_name in loader.list_records():
    record = loader[record_name]
    signal = record.get_signal(0)
    clean = preprocessor(signal)
    result = detector.run(clean)
    metrics = compute_metrics(clean, clean.annotation, result.annotation, Fiducials.R_PEAK)

    results.append(metrics.to_dict())

results_df = pd.DataFrame(results)

TP = np.sum(results_df["TP"])
FP = np.sum(results_df["FP"])
FN = np.sum(results_df["FN"])

sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
F1_score = (2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0)

print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"PPV (Precision):      {ppv:.4f}")
print(f"F1-score:             {F1_score:.4f}")

results_df

Sensitivity (Recall): 0.9717
PPV (Precision):      0.9962
F1-score:             0.9838


,Symbol,Reference,Predicted,TP,FP,FN,Sensitivity,Positive Predictive Value,F1 Score,Detection Error Rate,Mean Absolute Timing Error,Mean Absolute Timing Error Std,Mean Timing Error,Timing Error Std
0,R,2273,2272,2272,0,1,0.999560,1.000000,0.999780,0.000440,0.151604,0.630982,-0.124707,0.636844
1,R,1865,1864,1863,1,2,0.998928,0.999464,0.999195,0.001609,1.016878,1.338141,1.016878,1.338141
2,R,2187,2187,2175,12,12,0.994513,0.994513,0.994513,0.010974,25.117497,13.050984,23.653895,15.547028
3,R,2084,2078,2078,0,6,0.997121,1.000000,0.998558,0.002879,0.224575,0.757222,-0.195166,0.765330
4,R,2229,2042,2020,22,209,0.906236,0.989226,0.945914,0.103634,9.786854,14.738314,8.142189,15.706850
5,R,2572,2588,2545,43,27,0.989502,0.983385,0.986434,0.027216,0.813141,3.150076,0.479153,3.217855
6,R,2027,1972,1972,0,55,0.972866,1.000000,0.986247,0.027134,3.572233,10.947332,2.811584,11.166912
7,R,2137,2124,2123,1,14,0.993449,0.999529,0.996480,0.007019,4.606950,1.767355,4.520595,1.977817
8,R,1763,1756,1746,10,17,0.990357,0.994305,0.992327,0.015315,10.175640,15.763743,-9.195622,16.354809
9,R,2532,2525,2524,1,8,0.996840,0.999604,0.998220,0.003555,1.192992,3.368860,-0.906850,3.456887


In [4]:
detector = build_detector("wavelet")

results = []
for record_name in loader.list_records():
    record = loader[record_name]
    signal = record.get_signal(0)
    clean = preprocessor(signal)
    result = detector.run(clean)
    metrics = compute_metrics(clean, clean.annotation, result.annotation, Fiducials.R_PEAK, tolerance_ms=150)

    results.append(metrics.to_dict())

results_df = pd.DataFrame(results)

TP = np.sum(results_df["TP"])
FP = np.sum(results_df["FP"])
FN = np.sum(results_df["FN"])

sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
F1_score = (2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0)

print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"PPV (Precision):      {ppv:.4f}")
print(f"F1-score:             {F1_score:.4f}")

results_df

Sensitivity (Recall): 0.9976
PPV (Precision):      0.8889
F1-score:             0.9401


,Symbol,Reference,Predicted,TP,FP,FN,Sensitivity,Positive Predictive Value,F1 Score,Detection Error Rate,Mean Absolute Timing Error,Mean Absolute Timing Error Std,Mean Timing Error,Timing Error Std
0,R,2273,2274,2273,1,0,1.000000,0.999560,0.999780,0.000440,0.708804,5.168746,-0.058660,5.216790
1,R,1865,1877,1865,12,0,1.000000,0.993607,0.996793,0.006434,1.903485,2.956175,-1.903485,2.956175
2,R,2187,2197,2187,10,0,1.000000,0.995448,0.997719,0.004572,2.958136,10.615300,-2.851445,10.644455
3,R,2084,2085,2084,1,0,1.000000,0.999520,0.999760,0.000480,0.806409,1.260845,0.806409,1.260845
4,R,2229,2240,2228,12,1,0.999551,0.994643,0.997091,0.005832,5.599192,10.353171,-4.252693,10.975140
5,R,2572,2678,2565,113,7,0.997278,0.957804,0.977143,0.046656,15.740741,33.498526,-14.766082,33.939431
6,R,2027,2827,2026,801,1,0.999507,0.716661,0.834775,0.395659,3.830756,12.138348,-2.330811,12.513253
7,R,2137,2453,2130,323,7,0.996724,0.868325,0.928105,0.154422,6.653625,6.564917,-6.640584,6.578108
8,R,1763,2112,1593,519,170,0.903573,0.754261,0.822194,0.390811,41.126107,47.526228,-32.874730,53.566325
9,R,2532,2532,2532,0,0,1.000000,1.000000,1.000000,0.000000,2.654906,4.631509,-2.378445,4.779373
